# LIDC-IDRI DICOM Metadata Exploration

### Automating CT Protocol Classification with Large Language Models

**Purpose:**  
This notebook performs the initial acquisition and exploratory analysis of CT series metadata from the LIDC-IDRI collection available through The Cancer Imaging Archive (TCIA).

The goal of this stage is to understand the structure, completeness, and variability of the available DICOM metadata before defining a standardized protocol taxonomy. Particular attention will be given to fields related to protocol naming, scanner manufacturer, and acquisition characteristics.

### Initial Questions

1. What DICOM metadata fields are available at the series level?
2. How complete are the fields relevant to protocol classification?
3. How much variation exists in `SeriesDescription` and `ProtocolName`?
4. Are there identifiable differences in terminology across scanner manufacturers or models?
5. Does the available metadata support a small, clinically meaningful classification taxonomy?

### Data Source

- **Collection:** LIDC-IDRI
- **Modality:** CT
- **Source:** The Cancer Imaging Archive (TCIA)
- **Access:** TCIA/NBIA API via `tcia_utils`

In [1]:
"""
Starter script: pull a small sample of CT series metadata from LIDC-IDRI
via tcia_utils, to sanity-check real Series Description values before
finalizing the taxonomy.

"""

import pandas as pd
from tcia_utils import nbia

COLLECTION = "LIDC-IDRI"
MODALITY = "CT"

# --- 1. Pull series-level metadata for the collection, filtered to CT -------
# This returns a list of dicts (one per series) with fields like
# SeriesInstanceUID, SeriesDescription, Modality, Manufacturer,
# BodyPartExamined, PatientID, StudyInstanceUID, etc.

series_data = nbia.getSeries(collection=COLLECTION, modality=MODALITY)

df = pd.DataFrame(series_data)

print(f"Pulled {len(df)} series from {COLLECTION} ({MODALITY})")
print(df.columns.tolist())

# --- 2. Look at a small sample of the fields we actually care about ---------

cols_of_interest = [c for c in [
    "SeriesDescription",
    "StudyDescription",
    "ProtocolName",
    "Manufacturer",
    "ManufacturerModelName",
    "BodyPartExamined",
    "Modality",
] if c in df.columns]

sample = df[cols_of_interest].drop_duplicates().sample(
    n=min(30, len(df)), random_state=42
)

print("\nSample of real Series Description values:")
print(sample.to_string(index=False))

# --- 3. Save locally for reference while you build out the taxonomy --------

df.to_csv("lidc_idri_ct_series_metadata.csv", index=False)
print("\nSaved full metadata pull to lidc_idri_ct_series_metadata.csv")

# Next steps once you've looked at the sample:
#   - Check whether the taxonomy (body region x contrast status) actually
#     fits what you're seeing, or needs adjusting
#   - Repeat this pull for a second collection (e.g. a TCGA-* set) if you
#     want more body-region variety beyond chest/lung
#   - Move on to hand-labeling a 150-300 row validation subset

2026-09-26 00:50:41,255:INFO:Calling getSeries with parameters {'Collection': 'LIDC-IDRI', 'Modality': 'CT'}


Pulled 1018 series from LIDC-IDRI (CT)
['SeriesInstanceUID', 'StudyInstanceUID', 'Modality', 'SeriesDate', 'BodyPartExamined', 'SeriesNumber', 'AnnotationsFlag', 'Collection', 'Site', 'PatientID', 'Manufacturer', 'ManufacturerModelName', 'SoftwareVersions', 'ImageCount', 'MaxSubmissionTimestamp', 'LicenseName', 'LicenseURI', 'DataDescriptionURI', 'FileSize', 'DateReleased', 'StudyDate', 'Authorized', 'SeriesDescription', 'PatientSex', 'PatientAge', 'StudyDesc', 'ProtocolName', 'ReleasedStatus']

Sample of real Series Description values:
            SeriesDescription                                   ProtocolName       Manufacturer ManufacturerModelName BodyPartExamined Modality
                          NaN                                            NaN            SIEMENS          Sensation 16            CHEST       CT
               Recon 2: CHEST                            5.1  @ CHEST 5mm  @ GE MEDICAL SYSTEMS       LightSpeed Plus            CHEST       CT
                BOTTOM TO

## 2. Initial Metadata Profiling

Before defining a protocol taxonomy, the metadata will be profiled to assess
completeness, variability, and potential differences across scanner manufacturers.

This step will help determine which DICOM fields contain sufficient information
for protocol classification and which fields should be included in the
classification pipeline.

In [2]:
df.shape

(1018, 28)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1018 entries, 0 to 1017
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SeriesInstanceUID       1018 non-null   str    
 1   StudyInstanceUID        1018 non-null   str    
 2   Modality                1018 non-null   str    
 3   SeriesDate              1018 non-null   str    
 4   BodyPartExamined        1018 non-null   str    
 5   SeriesNumber            884 non-null    float64
 6   AnnotationsFlag         918 non-null    str    
 7   Collection              1018 non-null   str    
 8   Site                    1018 non-null   str    
 9   PatientID               1018 non-null   str    
 10  Manufacturer            1018 non-null   str    
 11  ManufacturerModelName   1018 non-null   str    
 12  SoftwareVersions        936 non-null    str    
 13  ImageCount              1018 non-null   int64  
 14  MaxSubmissionTimestamp  1018 non-null   str    
 15

### Missing Metadata

To assess metadata completeness, missing values were counted for each column
and converted to percentages relative to the total number of CT series.

In [4]:
missing_values = df.isna().sum()

total_rows = len(df)

percentage_missing = missing_values / total_rows * 100

percentage_missing

SeriesInstanceUID          0.000000
StudyInstanceUID           0.000000
Modality                   0.000000
SeriesDate                 0.000000
BodyPartExamined           0.000000
SeriesNumber              13.163065
AnnotationsFlag            9.823183
Collection                 0.000000
Site                       0.000000
PatientID                  0.000000
Manufacturer               0.000000
ManufacturerModelName      0.000000
SoftwareVersions           8.055010
ImageCount                 0.000000
MaxSubmissionTimestamp     0.000000
LicenseName                0.000000
LicenseURI                 0.000000
DataDescriptionURI         0.000000
FileSize                   0.000000
DateReleased               0.000000
StudyDate                  0.000000
Authorized                 0.000000
SeriesDescription         69.351670
PatientSex                72.102161
PatientAge                79.666012
StudyDesc                 53.929273
ProtocolName              80.844794
ReleasedStatus            95

### Scanner Manufacturer Distribution

The distribution of scanner manufacturers was examined to determine whether
the collection contains sufficient vendor diversity to investigate differences
in DICOM metadata terminology.

In [5]:
# What unique manufacturers exist?

df["Manufacturer"].unique()

<ArrowStringArray>
['GE MEDICAL SYSTEMS', 'SIEMENS', 'TOSHIBA', 'Philips']
Length: 4, dtype: str

In [6]:
# How many times does each unique manufacturer occur?

df["Manufacturer"].value_counts()

Manufacturer
GE MEDICAL SYSTEMS    670
SIEMENS               205
Philips                74
TOSHIBA                69
Name: count, dtype: int64

In [7]:
# How many series came from each scanner model.

df.groupby("ManufacturerModelName")["Manufacturer"].count()

ManufacturerModelName
Aquilion              69
Brilliance 16P        54
Brilliance 40          9
Brilliance 64          5
Brilliance16           6
Definition             3
Emotion 6             30
Emotion Duo            1
LightSpeed Plus       56
LightSpeed Power      10
LightSpeed Pro 16     80
LightSpeed QX/i       98
LightSpeed Ultra     162
LightSpeed VCT        62
LightSpeed16         202
Sensation 16         117
Sensation 64          54
Name: Manufacturer, dtype: int64

In [8]:
df.groupby("Manufacturer")["ManufacturerModelName"].count()

Manufacturer
GE MEDICAL SYSTEMS    670
Philips                74
SIEMENS               205
TOSHIBA                69
Name: ManufacturerModelName, dtype: int64

In [9]:
df.groupby("Manufacturer")["ManufacturerModelName"].unique()

Manufacturer
GE MEDICAL SYSTEMS    [LightSpeed Plus, LightSpeed16, LightSpeed QX/...
Philips               [Brilliance 16P, Brilliance 40, Brilliance16, ...
SIEMENS               [Sensation 64, Sensation 16, Emotion 6, Emotio...
TOSHIBA                                                      [Aquilion]
Name: ManufacturerModelName, dtype: object

In [10]:
# What terminology actually appears in SeriesDescription, ProtocolName, and StudyDesc?

df["SeriesDescription"].value_counts()

SeriesDescription
Recon 2: ACRIN LARGE               44
CHEST                              38
NLST TLC VOL B30F                  30
ThoraxRoutine  3.0  B31s           24
Recon 2: CHEST                     23
Recon 3: C-A-P                     23
BOTTOM TO TOP                      22
Recon 2:                           13
ChestRoutine  3.0  B31f            12
Recon 2: C-A-P                     11
ThorRoutine  2.0  B70f              8
Recon 2: LOW DOSE THROUGH LUNGS     7
ThorRoutine  3.0  B31f              6
NLST_0.6_ B30f                      5
ACRIN LARGE                         4
ThoraxRoutine  3.0  B31f            3
Recon 3:                            2
Recon 3: CHEST                      2
CT, PE PROTOCOL BOTTOM TO TOP       2
Recon 2: ACRIN SMALL                2
Routine Chest  2.0  B70f            2
Recon 3: CHEST-ABD                  2
ChestRoutine  2.0  B70f             2
Routine Chest  3.0  B31s            2
CAP  5.0  B30s                      2
Recon 2: CHEST-ABD              

In [11]:
series_counts = df["SeriesDescription"].value_counts()

series_counts.head(20)

SeriesDescription
Recon 2: ACRIN LARGE               44
CHEST                              38
NLST TLC VOL B30F                  30
ThoraxRoutine  3.0  B31s           24
Recon 2: CHEST                     23
Recon 3: C-A-P                     23
BOTTOM TO TOP                      22
Recon 2:                           13
ChestRoutine  3.0  B31f            12
Recon 2: C-A-P                     11
ThorRoutine  2.0  B70f              8
Recon 2: LOW DOSE THROUGH LUNGS     7
ThorRoutine  3.0  B31f              6
NLST_0.6_ B30f                      5
ACRIN LARGE                         4
ThoraxRoutine  3.0  B31f            3
Recon 3:                            2
Recon 3: CHEST                      2
CT, PE PROTOCOL BOTTOM TO TOP       2
Recon 2: ACRIN SMALL                2
Name: count, dtype: int64

In [12]:
df["SeriesDescription"].nunique()

46

**Initial observation:** Of the 1,018 CT series, 312 contain a populated
`SeriesDescription`, representing 46 unique descriptions. The descriptions
show variation in terminology and formatting, including anatomical labels,
reconstruction terminology, acquisition direction, and protocol-specific
language. This supports further investigation into whether these heterogeneous
descriptions can be mapped to a smaller standardized taxonomy.

In [13]:
# What are the 20 most common ProtocolName values?

protocol_counts = df["ProtocolName"].value_counts()
protocol_counts.head(20)

ProtocolName
5.1  @ CHEST @                                         33
5.11 FULL CHECT F/U                                    28
5.13 LUNG SCREEN                                       18
5.7 @ CHEST PE BOTTOM TO TOP @                         16
5.1  @ CHEST   @                                       15
5.7 @ C-A-P  @                                         10
5.11 SCREENING CHEST                                    6
5.3 LUNG SCREENING                                      6
5.3 @ C-A-P 5mm @                                       5
5.10 Full Chest F/U                                     5
5.4 @ C-A-P LYM. HEAD FIRST @                           4
5.9 @ CHEST PE BOTTOM TO TOP @                          4
5.3 @ C-A-P  @                                          4
5.20 LUNG BX. PRONE                                     4
5.1  @ CHEST 5mm  @                                     3
5.5 @ CHEST-ABD  @                                      3
5.19 LUNG BX. SUPINE                                    3
5

In [14]:
# How many unique non-null ProtocolName values exist?

df["ProtocolName"].nunique()

38

Initial observation: ProtocolName is sparsely populated, with 195 non-null values across the 1,018 CT series. Among those records, 38 unique protocol names are present. The values include differences in abbreviations, capitalization, acquisition positioning, contrast information, and terminology for clinically related examinations.

In [15]:
# Which series have all three fields missing?

text_fields = [
    "SeriesDescription",
    "ProtocolName",
    "StudyDesc"
]

all_text_missing = df[text_fields].isna().all(axis=1)

print(all_text_missing)

0        True
1        True
2        True
3        True
4        True
        ...  
1013    False
1014    False
1015    False
1016    False
1017    False
Length: 1018, dtype: bool


In [16]:
# How many series have ALL THREE fields missing?

total_missing = all_text_missing.sum()

print(total_missing)


445


In [17]:
# How many series have AT LEAST ONE field populated?

at_least_one_text = total_rows - total_missing

In [18]:
print(at_least_one_text)

573


In [19]:
text_coverage_pct = at_least_one_text / total_rows * 100
text_coverage_pct

np.float64(56.286836935166995)

Initial observation: Although the individual descriptive metadata fields are sparsely populated, 573 of 1,018 CT series (approximately 56%) contain at least one of SeriesDescription, ProtocolName, or StudyDesc. This suggests that combining available descriptive fields may provide substantially better coverage than relying on any single field alone.

In [20]:
df[text_fields].notna()

,SeriesDescription,ProtocolName,StudyDesc
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
1013,False,True,True
1014,False,True,True
1015,False,True,True
1016,False,True,True


In [21]:
df[text_fields].notna().sum()

SeriesDescription    312
ProtocolName         195
StudyDesc            469
dtype: int64

In [22]:
df[text_fields].notna().value_counts()

SeriesDescription  ProtocolName  StudyDesc
False              False         False        445
                                 True         179
True               True          True         113
                   False         False        104
                                 True          95
False              True          True          82
Name: count, dtype: int64

Initial observation: Descriptive metadata availability varies considerably across the collection. Of the 1,018 CT series, 445 contain none of the three selected descriptive fields, while 573 contain at least one. The available fields also occur in several combinations rather than following a uniform schema. Notably, ProtocolName is only present when StudyDesc is also populated. This variability supports using a classification approach capable of incorporating whichever descriptive metadata fields are available for a given series.

## 3. Exploring Protocol Terminology and Developing the Taxonomy

In [23]:
# Identify series with at least one usable descriptive metadata field
usable_mask = df[text_fields].notna().any(axis=1)

# Retain series eligible for taxonomy development and classification
usable_df = df[usable_mask].copy()

usable_df.shape

(573, 28)

In [24]:
usable_df[text_fields].head(3)

,SeriesDescription,ProtocolName,StudyDesc
26,Recon 2: C-A-P,NaN,CT/C-A-P
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST


In [25]:
usable_df[text_fields].fillna("")

,SeriesDescription,ProtocolName,StudyDesc
26,Recon 2: C-A-P,,CT/C-A-P
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
29,Recon 2: CHEST,,CT THORAX W/CONTRAST
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST
...,...,...,...
1013,,5.11 FULL CHECT F/U,CT CHEST O CONTR
1014,,5.13 LUNG SCREEN,CT LUNG SCREEN
1015,,5.13 LUNG SCREEN,CT LUNG SCREEN
1016,,5.3 LUNG SCREENING,CT CHEST WO CONTRAST


In [26]:
text_data = usable_df[text_fields].fillna("")

text_data.head()

,SeriesDescription,ProtocolName,StudyDesc
26,Recon 2: C-A-P,,CT/C-A-P
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
29,Recon 2: CHEST,,CT THORAX W/CONTRAST
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST


In [27]:
text_data["combined_text"] = (
    "SeriesDescription: " + text_data["SeriesDescription"]
)

text_data[["SeriesDescription", "combined_text"]].head()

,SeriesDescription,combined_text
26,Recon 2: C-A-P,SeriesDescription: Recon 2: C-A-P
27,Recon 2: CHEST,SeriesDescription: Recon 2: CHEST
28,Recon 2: CHEST,SeriesDescription: Recon 2: CHEST
29,Recon 2: CHEST,SeriesDescription: Recon 2: CHEST
30,Recon 2: CHEST,SeriesDescription: Recon 2: CHEST


In [28]:
text_data["combined_text"] = (
    "SeriesDescription: " + text_data["SeriesDescription"]
    + " | ProtocolName: " + text_data["ProtocolName"]
    + " | StudyDesc: " + text_data["StudyDesc"]
)
text_data["combined_text"].head()

26    SeriesDescription: Recon 2: C-A-P | ProtocolNa...
27    SeriesDescription: Recon 2: CHEST | ProtocolNa...
28    SeriesDescription: Recon 2: CHEST | ProtocolNa...
29    SeriesDescription: Recon 2: CHEST | ProtocolNa...
30    SeriesDescription: Recon 2: CHEST | ProtocolNa...
Name: combined_text, dtype: str

In [29]:
print(text_data["combined_text"].head(5).to_string(index=False))

SeriesDescription: Recon 2: C-A-P | ProtocolNam...
SeriesDescription: Recon 2: CHEST | ProtocolNam...
SeriesDescription: Recon 2: CHEST | ProtocolNam...
SeriesDescription: Recon 2: CHEST | ProtocolNam...
SeriesDescription: Recon 2: CHEST | ProtocolNam...


### Working Data Flow

The metadata preparation steps completed so far can be summarized as:

**1,018 CT series retrieved from LIDC-IDRI**  
↓  
**573 series with at least one usable descriptive metadata field**  
(`SeriesDescription`, `ProtocolName`, or `StudyDesc`)  
↓  
**Missing text values temporarily represented as empty strings (`""`)**  
↓  
**Three descriptive fields combined into a single text representation**  
↓  
**`combined_text` created for each usable CT series**

The `combined_text` field provides a consistent textual input that can later be used for both the fuzzy-string-matching baseline and the LLM classifier.

**Current analysis population: 573 CT series**

In [30]:
combined_counts = text_data["combined_text"].value_counts()

combined_counts.head(20)

combined_text
SeriesDescription:  | ProtocolName:  | StudyDesc: CT LUNG SCREEN                                                       120
SeriesDescription: Recon 2: ACRIN LARGE | ProtocolName:  | StudyDesc: CHEST                                             30
SeriesDescription: NLST TLC VOL B30F | ProtocolName:  | StudyDesc:                                                      30
SeriesDescription: ThoraxRoutine  3.0  B31s | ProtocolName:  | StudyDesc:                                               24
SeriesDescription: CHEST | ProtocolName: 5.1  @ CHEST @ | StudyDesc: CT THORAX W/CONTRAST                               21
SeriesDescription:  | ProtocolName:  | StudyDesc: CHEST                                                                 21
SeriesDescription:  | ProtocolName: 5.11 FULL CHECT F/U | StudyDesc: CT CHEST O CONTR                                   16
SeriesDescription:  | ProtocolName:  | StudyDesc: CT CHEST O CONTR                                                      13
Se

In [31]:
# How many unique combined text representations exist?

text_data["combined_text"].nunique()

143

In [32]:
# What are the 20 most common StudyDesc values?

text_data["StudyDesc"].value_counts().head(20)

StudyDesc
CT LUNG SCREEN            135
                          104
CHEST                      74
CT THORAX W/CONTRAST       71
CT CHEST O CONTR           32
CT GUIDED LUNG B           22
CT CHEST WO CONTRAST       19
CT THORAX WO IV CONTRA     18
CT ANGIO CHEST (NON-CO     12
CT ANGIO CHEST W&W/O C     11
CT THORAX W/O CONTRAST     10
CT CAP                      7
CT CHEST W/O CON            5
CT CHEST W/ CONT            5
CAP                         4
CT GUIDED LUNG BIOPSY       4
CT CH/AB/PELVIS             2
CT C/A                      2
CHEST/ABD/PELVIS            2
POST CHEST                  2
Name: count, dtype: int64

In [33]:
# What are the 20 most common ProtocolName values?

text_data["ProtocolName"].value_counts().head(20)

ProtocolName
                                          378
5.1  @ CHEST @                             33
5.11 FULL CHECT F/U                        28
5.13 LUNG SCREEN                           18
5.7 @ CHEST PE BOTTOM TO TOP @             16
5.1  @ CHEST   @                           15
5.7 @ C-A-P  @                             10
5.11 SCREENING CHEST                        6
5.3 LUNG SCREENING                          6
5.3 @ C-A-P 5mm @                           5
5.10 Full Chest F/U                         5
5.4 @ C-A-P LYM. HEAD FIRST @               4
5.9 @ CHEST PE BOTTOM TO TOP @              4
5.3 @ C-A-P  @                              4
5.20 LUNG BX. PRONE                         4
5.1  @ CHEST 5mm  @                         3
5.5 @ CHEST-ABD  @                          3
5.19 LUNG BX. SUPINE                        3
5.1 ROUTINE CHEST WITH IV CONTRAST 100      2
5.8 @CHEST P E BOTTOM TO TOP@               2
Name: count, dtype: int64

In [34]:
# What are the 20 most common SeriesDescription values?

text_data["SeriesDescription"].value_counts().head(20)

SeriesDescription
                                   261
Recon 2: ACRIN LARGE                44
CHEST                               38
NLST TLC VOL B30F                   30
ThoraxRoutine  3.0  B31s            24
Recon 2: CHEST                      23
Recon 3: C-A-P                      23
BOTTOM TO TOP                       22
Recon 2:                            13
ChestRoutine  3.0  B31f             12
Recon 2: C-A-P                      11
ThorRoutine  2.0  B70f               8
Recon 2: LOW DOSE THROUGH LUNGS      7
ThorRoutine  3.0  B31f               6
NLST_0.6_ B30f                       5
ACRIN LARGE                          4
ThoraxRoutine  3.0  B31f             3
Recon 3:                             2
Recon 3: CHEST                       2
CT, PE PROTOCOL BOTTOM TO TOP        2
Name: count, dtype: int64

### Preliminary Terminology Assessment and Candidate Taxonomy

Exploration of the 573 usable CT series demonstrates substantial heterogeneity across the three descriptive DICOM metadata fields. `StudyDesc` appears to provide the most consistently interpretable clinical information, including terminology related to lung screening, contrast-enhanced and noncontrast chest CT, CT angiography, and CT-guided biopsy. `ProtocolName` is substantially less complete but contains useful protocol-specific terminology when available, including screening, pulmonary embolism (PE), follow-up, and lung biopsy descriptions. `SeriesDescription` contains a mixture of clinical and technical information, including anatomical descriptions, reconstruction labels, slice thicknesses, and reconstruction kernels.

Because similar clinical examinations can be represented using different terminology across these fields, exact text values should not be treated as distinct protocol categories. Instead, the initial taxonomy will group semantically and clinically related descriptions into a small number of standardized classes.

#### Candidate Taxonomy

| Candidate Category | Example Terminology Observed | Initial Interpretation / Questions |
|---|---|---|
| **Lung Screening / Low-Dose Screening** | `CT LUNG SCREEN`, `LUNG SCREENING`, `NLST`, `LOW DOSE THROUGH LUNGS` | Likely strong candidate. Determine whether all NLST/low-dose terminology can reliably be interpreted as screening. |
| **Routine Chest – Noncontrast** | `CT CHEST WO CONTRAST`, `CT THORAX WO IV CONTRA`, `CT THORAX W/O CONTRAST` | Strong explicit noncontrast examples. Determine how generic `CHEST` descriptions should be handled when contrast status is unavailable. |
| **Routine Chest – Contrast** | `CT THORAX W/CONTRAST`, `CT CHEST W/ CONT`, `ROUTINE CHEST WITH IV CONTRAST` | Strong explicit contrast examples. Determine whether this should remain separate from noncontrast routine chest. |
| **CT Angiography / PE** | `CT ANGIO CHEST`, `CHEST PE`, `PE PROTOCOL` | Likely clinically distinct from routine contrast-enhanced chest CT. |
| **CT-Guided Biopsy / Procedure** | `CT GUIDED LUNG B`, `CT GUIDED LUNG BIOPSY`, `LUNG BX. PRONE`, `LUNG BX. SUPINE` | Likely distinct procedural category. Determine whether positioning/reconstruction descriptions should map to the same examination. |
| **Multiregion / Chest-Abdomen-Pelvis** | `CT CAP`, `CHEST/ABD/PELVIS`, `C-A-P`, `CT C/A` | Potential category, but frequency and relevance to the primary thoracic taxonomy should be evaluated. |
| **UNKNOWN / INSUFFICIENT INFORMATION** | Missing descriptive fields, generic reconstruction terminology, or descriptions without enough clinical information | Abstention option rather than forcing an unsupported classification. |

#### Taxonomy Development Questions

Before finalizing the taxonomy, the following decisions will need to be made:

- Should routine contrast-enhanced and noncontrast chest CT be represented as separate categories?
- Should generic terms such as `CHEST` be classified as routine chest or treated as insufficiently specific?
- Can `NLST`, low-dose, and lung-screening terminology be reliably grouped into one screening category?
- Should CT angiography and pulmonary embolism protocols be combined into one category?
- Should CT-guided biopsy/procedural examinations be included in the primary taxonomy?
- Are multiregion examinations such as chest-abdomen-pelvis sufficiently represented to justify their own category?
- Which technical reconstruction terms provide useful protocol information, and which should be ignored during manual labeling?
- Under what conditions should a series receive an `UNKNOWN` or `INSUFFICIENT_INFORMATION` designation?

**Current working goal:** Begin with approximately 4–5 clinically defensible protocol categories, plus an abstention option, and expand only if the available labeled data support additional categories.

In [35]:
review_cols = [
    "SeriesDescription",
    "ProtocolName",
    "StudyDesc",
    "combined_text"
]

text_data[review_cols].head(20)

,SeriesDescription,ProtocolName,StudyDesc,combined_text
26,Recon 2: C-A-P,,CT/C-A-P,SeriesDescription: Recon 2: C-A-P | ProtocolNa...
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,SeriesDescription: Recon 2: CHEST | ProtocolNa...
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,SeriesDescription: Recon 2: CHEST | ProtocolNa...
29,Recon 2: CHEST,,CT THORAX W/CONTRAST,SeriesDescription: Recon 2: CHEST | ProtocolNa...
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST,SeriesDescription: Recon 2: CHEST | ProtocolNa...
31,Recon 2: CHEST,,"CT, LUNG, MED, PLEURA",SeriesDescription: Recon 2: CHEST | ProtocolNa...
32,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/O CONTRAST,SeriesDescription: Recon 2: CHEST | ProtocolNa...
33,,,CT THORAX W/O CONTRAST,SeriesDescription: | ProtocolName: | StudyDe...
34,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT CH/AB/PELVIS,SeriesDescription: Recon 2: C-A-P | ProtocolNa...
35,,,CT THORAX W/CONTRAST,SeriesDescription: | ProtocolName: | StudyDe...


### Draft Manual Labeling Rules

Manual reference labels will be assigned using the available `SeriesDescription`,
`ProtocolName`, and `StudyDesc` metadata. Labels should only be assigned when the
available metadata provide sufficient evidence for the category. Reconstruction
terminology alone will not be assumed to identify the clinical examination.

#### CHEST_CONTRAST
Assign when the metadata explicitly indicate a routine chest/thorax CT performed
with intravenous contrast.

Examples:
- `CT THORAX W/CONTRAST`
- `CT CHEST W/ CONT`
- `ROUTINE CHEST WITH IV CONTRAST`

Do not infer contrast status from `CHEST` or `THORAX` alone.

#### CHEST_NONCONTRAST
Assign when the metadata explicitly indicate a routine chest/thorax CT without
intravenous contrast.

Examples:
- `CT CHEST WO CONTRAST`
- `CT THORAX WO IV CONTRA`
- `CT THORAX W/O CONTRAST`
- `CT CHEST O CONTR`

#### LUNG_SCREENING
Assign when the metadata explicitly indicate lung cancer screening or when
terminology is determined to be a reliable indicator of a screening examination.

Potential indicators:
- `CT LUNG SCREEN`
- `LUNG SCREENING`
- `NLST`
- `LOW DOSE THROUGH LUNGS`

The reliability of `NLST` and other indirect screening terminology should be
verified before being used as definitive labeling criteria.

#### CTA_PE
Assign when the metadata explicitly indicate CT angiography of the chest or a
pulmonary embolism protocol.

Potential indicators:
- `CT ANGIO CHEST`
- `CHEST PE`
- `PE PROTOCOL`

#### MULTIREGION_CAP
Assign when the examination explicitly includes chest, abdomen, and pelvis or
another defined multiregion examination that includes the chest.

Potential indicators:
- `CT CAP`
- `C-A-P`
- `CH/ABD/PELVIS`
- `CHEST/ABD/PELVIS`
- `T/A/P`

#### UNKNOWN / INSUFFICIENT_INFORMATION
Assign when the available metadata do not provide enough evidence to distinguish
among the defined clinical categories.

Examples may include:
- Generic `CHEST` descriptions without sufficient protocol information
- Reconstruction terminology without clear clinical intent
- Missing or conflicting descriptive fields
- Descriptions outside the scope of the final taxonomy

### General Labeling Principle

Labels will be based on explicit or clinically defensible evidence contained in
the metadata rather than assumptions about the examination. Ambiguous cases will
be assigned to `UNKNOWN / INSUFFICIENT_INFORMATION` rather than forced into a
clinical category.

In [36]:
# Biopsy terminology count

text_data["combined_text"].str.contains("BIOPSY").sum()

np.int64(4)

In [37]:
biopsy_mask = (
    text_data["combined_text"].str.contains("BIOPSY") |
    text_data["combined_text"].str.contains("BX")
)

In [38]:
biopsy_mask.sum()

np.int64(11)

In [39]:
text_data[biopsy_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
147,,8.2 LUNG BX PRONE,CT GUIDED LUNG B
587,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B
591,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B
594,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B
597,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B
605,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B
609,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B
611,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG BIOPSY
1000,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY
1001,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY


#### CT-Guided Lung Biopsy Terminology

A terminology search using `BIOPSY` or the abbreviation `BX` identified 11
candidate CT-guided lung biopsy series. Manual inspection confirmed that `BX`
was used in biopsy-related protocol names such as `LUNG BX. PRONE` and
`LUNG BX. SUPINE`, while `StudyDesc` contained descriptions such as
`CT GUIDED LUNG B` and `CT GUIDED LUNG BIOPSY`.

Several records contained potentially conflicting metadata. For example,
`ProtocolName` values associated with follow-up or lung screening occurred
alongside a `StudyDesc` explicitly identifying CT-guided lung biopsy. These
cases demonstrate the importance of considering multiple metadata fields
rather than relying on a single field or exact text match.

Because only 11 candidate biopsy series were identified, this category will
remain provisional until class frequencies for the other candidate taxonomy
categories are evaluated.

In [40]:
cte_pe_mask_broad = (
    text_data["combined_text"].str.contains("PE") |
    text_data["combined_text"].str.contains("ANGIO")
)
cte_pe_mask_broad.sum()

np.int64(34)

In [41]:
text_data[cte_pe_mask_broad][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
34,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT CH/AB/PELVIS
36,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CH/ABD/PELVIS
38,,,CT CH/ABD/PELVIS
52,,5.3 @ C-A-P 5mm @,CHEST/ABD/PELVIS
93,BOTTOM TO TOP,,CT ANGIO CHEST W&W/O C
112,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
113,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST W&W/O C
114,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
118,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
125,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO


In [42]:
cte_pe_mask= (
    text_data["combined_text"].str.contains(r"\bPE\b", regex=True) |
    text_data["combined_text"].str.contains("ANGIO")
)
cte_pe_mask.sum()

np.int64(25)

In [43]:
text_data[cte_pe_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
93,BOTTOM TO TOP,,CT ANGIO CHEST W&W/O C
112,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
113,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST W&W/O C
114,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
118,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
125,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
129,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
130,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
134,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO
136,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO


In [44]:
screening_mask = (
    text_data["combined_text"].str.contains("SCREEN")|
    text_data["combined_text"].str.contains("NLST")
)

text_data[screening_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
146,,,CT LUNG SCREEN
148,,,CT LUNG SCREEN
149,,,CT LUNG SCREEN
150,,,CT LUNG SCREEN
151,,,CT LUNG SCREEN
...,...,...,...
1011,,5.11 SCREENING CHEST,CT CHEST WO CONTRAST
1014,,5.13 LUNG SCREEN,CT LUNG SCREEN
1015,,5.13 LUNG SCREEN,CT LUNG SCREEN
1016,,5.3 LUNG SCREENING,CT CHEST WO CONTRAST


In [45]:
text_data["combined_text"].str.contains("SCREEN").sum()

np.int64(151)

In [46]:
text_data["combined_text"].str.contains("NLST").sum()

np.int64(35)

In [47]:
nlst_only_mask = (
    text_data["combined_text"].str.contains("NLST")& ~
    text_data["combined_text"].str.contains("SCREEN")
)

text_data[nlst_only_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
452,NLST TLC VOL B30F,,
455,NLST TLC VOL B30F,,
457,NLST TLC VOL B30F,,
463,NLST TLC VOL B30F,,
464,NLST TLC VOL B30F,,
465,NLST TLC VOL B30F,,
467,NLST TLC VOL B30F,,
468,NLST TLC VOL B30F,,
469,NLST TLC VOL B30F,,
470,NLST TLC VOL B30F,,


In [48]:
nlst_only_mask.sum()

np.int64(35)

#### Lung Screening Terminology

The initial screening search identified 186 candidate series. Of these, 151
contained `SCREEN`, while 35 contained `NLST` without also containing `SCREEN`.
Manual inspection of the NLST-only records showed consistent SeriesDescription
patterns such as `NLST TLC VOL B30F` and `NLST_0.6_ B30f`, with ProtocolName and
StudyDesc generally unavailable.

These findings indicate that lung-screening context may be represented through
different terminology across the metadata and that relying only on the literal
term `SCREEN` would miss relevant series. Because some NLST descriptions also
contain reconstruction-specific terminology, NLST will be treated as strong
evidence of screening context while the final labeling rules will distinguish
clinical protocol information from reconstruction-level descriptions.

In [49]:
noncontrast_mask = text_data["combined_text"].str.contains("WO CONTRAST")

text_data[noncontrast_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
958,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
960,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
961,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
962,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
964,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
967,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
970,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
972,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
977,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
978,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST


In [50]:
screening_noncontrast_mask = (
    screening_mask & noncontrast_mask
)

screening_noncontrast_mask.sum()

np.int64(9)

In [51]:
noncontrast_only_mask = (
    text_data["combined_text"].str.contains("WO CONTRAST") & ~
    text_data["combined_text"].str.contains("SCREEN")
)

noncontrast_only_mask.sum()

np.int64(10)

In [52]:
noncontrast_only_mask.info()

<class 'pandas.Series'>
Index: 573 entries, 26 to 1017
Series name: combined_text
Non-Null Count  Dtype
--------------  -----
573 non-null    bool 
dtypes: bool(1)
memory usage: 5.0 KB


In [53]:
noncontrast_only_mask.shape

(573,)

In [54]:
noncontrast_only_mask.describe()

count       573
unique        2
top       False
freq        563
Name: combined_text, dtype: object

In [55]:
noncontrast_only_mask = (
    noncontrast_mask & ~screening_mask
)

In [56]:
text_data[noncontrast_only_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
958,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
960,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
962,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
977,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
978,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
980,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST
983,,5.10 Full Chest F/U,CT CHEST WO CONTRAST
986,,5.10 Full Chest F/U,CT CHEST WO CONTRAST
987,,5.10 Full Chest F/U,CT CHEST WO CONTRAST
1012,,5.10 Full Chest F/U,CT CHEST WO CONTRAST


In [57]:
text_data["combined_text"].str.contains("WO IV").sum()

np.int64(18)

In [58]:
text_data["combined_text"].str.contains("O CONTR").sum()

np.int64(61)

In [59]:
o_contr_mask = (
    text_data["combined_text"].str.contains("O CONTR")
)

In [60]:
text_data[o_contr_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
32,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/O CONTRAST
33,,,CT THORAX W/O CONTRAST
116,Recon 2: CHEST,,CT THORAX W/O CONTRAST
131,,,CT THORAX W/O CONTRAST
132,,,CT THORAX W/O CONTRAST
...,...,...,...
1006,,5.11 FULL CHECT F/U,CT CHEST O CONTR
1011,,5.11 SCREENING CHEST,CT CHEST WO CONTRAST
1012,,5.10 Full Chest F/U,CT CHEST WO CONTRAST
1013,,5.11 FULL CHECT F/U,CT CHEST O CONTR


In [61]:
text_data[o_contr_mask]["StudyDesc"].unique()

<ArrowStringArray>
['CT THORAX W/O CONTRAST', 'CT CHEST O CONTR', 'CT CHEST WO CONTRAST']
Length: 3, dtype: str

In [62]:
text_data[o_contr_mask]["StudyDesc"].value_counts()

StudyDesc
CT CHEST O CONTR          32
CT CHEST WO CONTRAST      19
CT THORAX W/O CONTRAST    10
Name: count, dtype: int64

In [63]:
text_data["combined_text"].str.contains("W/CONTRAST")

26      False
27       True
28       True
29       True
30       True
        ...  
1013    False
1014    False
1015    False
1016    False
1017    False
Name: combined_text, Length: 573, dtype: bool

In [64]:
w_contrast_mask = text_data["combined_text"].str.contains("W/CONTRAST")

text_data[w_contrast_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
29,Recon 2: CHEST,,CT THORAX W/CONTRAST
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST
35,,,CT THORAX W/CONTRAST
...,...,...,...
409,Recon 3: C-A-P,5.3 @ C-A-P @,CT THORAX W/CONTRAST
413,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
419,CHEST,5.7 @ CHEST HEAD FIRST @,CT THORAX W/CONTRAST
421,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST


In [65]:
text_data[w_contrast_mask]["StudyDesc"].unique()

<ArrowStringArray>
['CT THORAX W/CONTRAST', 'CT CAP W/CONTRAST']
Length: 2, dtype: str

In [66]:
text_data[w_contrast_mask]["StudyDesc"].value_counts()

StudyDesc
CT THORAX W/CONTRAST    71
CT CAP W/CONTRAST        1
Name: count, dtype: int64

In [67]:
text_data["combined_text"].str.contains("CAP")

26      False
27      False
28      False
29      False
30      False
        ...  
1013    False
1014    False
1015    False
1016    False
1017    False
Name: combined_text, Length: 573, dtype: bool

In [68]:
text_data["combined_text"].str.contains("CAP").sum()

np.int64(20)

In [69]:
cap_mask = text_data["combined_text"].str.contains("CAP")

In [70]:
text_data[cap_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
100,Recon 2: C-A-P,,CAP
127,,,CT CAP
133,,,CT CAP W/CONTRAST
268,Recon 3: C-A-P,5.4 @ C-A-P DFOV 40 and above,CAP W/CON
355,Recon 3:,5.5 CT CAP,CT CH/AB/PEL W/O
364,Recon 3: C-A-P,5.7 @ C-A-P @,CT CAP
367,Recon 3: C-A-P,5.7 @ C-A-P @,CAP
376,Recon 3: C-A-P,5.10 @ C-A-P LYM. HEAD FIRST 42 FOV AND ABOVE @,CAP
378,Recon 3: C-A-P,5.7 @ C-A-P @,CAP
379,Recon 3: C-A-P,5.3 @ C-A-P @,CT CAP


In [71]:
cap_mask.sum()

np.int64(20)

In [72]:
text_data["combined_text"].str.contains("C-A-P").sum()

np.int64(35)

In [73]:
text_data["combined_text"].str.contains("C/A/P").sum()

np.int64(5)

In [74]:
text_data["combined_text"].str.contains("PELV").sum()

np.int64(6)

In [75]:
pelv_mask = text_data["combined_text"].str.contains("PELV")

text_data[pelv_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
34,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT CH/AB/PELVIS
36,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CH/ABD/PELVIS
38,,,CT CH/ABD/PELVIS
52,,5.3 @ C-A-P 5mm @,CHEST/ABD/PELVIS
288,Recon 2: C-A-P,,CHEST/ABD/PELVIS
372,Recon 3: C-A-P,5.7 @ C-A-P @,CT CH/AB/PELVIS


In [76]:
text_data["combined_text"].str.contains("PELV").sum()

np.int64(6)

In [77]:
cap_expanded_mask = (
    text_data["combined_text"].str.contains("CAP") |
    text_data["combined_text"].str.contains("C-A-P") |
    text_data["combined_text"].str.contains("C/A/P") |
    text_data["combined_text"].str.contains("PELV")
)

cap_expanded_mask.sum()

np.int64(43)

In [78]:
text_data[cap_expanded_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
26,Recon 2: C-A-P,,CT/C-A-P
34,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT CH/AB/PELVIS
36,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CH/ABD/PELVIS
37,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,T/A/P W/C
38,,,CT CH/ABD/PELVIS
41,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT THORAX W/CONTRAST
46,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT-CH/A/P
48,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,CT THORAX W/CONTRAST
50,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,C/A/P POST
52,,5.3 @ C-A-P 5mm @,CHEST/ABD/PELVIS


In [79]:
generic_chest_mask = (
    text_data["StudyDesc"].str.strip() == "CHEST"
)

generic_chest_mask.sum()

np.int64(74)

In [80]:
text_data[generic_chest_mask]["SeriesDescription"].value_counts()

SeriesDescription
Recon 2: ACRIN LARGE               30
                                   21
Recon 2:                           13
Recon 2: LOW DOSE THROUGH LUNGS     4
ACRIN LARGE                         4
Recon 2: ACRIN SMALL                1
LOW DOSE THROUGH LUNGS              1
Name: count, dtype: int64

In [81]:
text_data[generic_chest_mask]["ProtocolName"].value_counts()

ProtocolName
    74
Name: count, dtype: int64

In [82]:
text_data[generic_chest_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
216,,,CHEST
295,Recon 2: ACRIN LARGE,,CHEST
296,Recon 2: ACRIN LARGE,,CHEST
297,Recon 2: ACRIN LARGE,,CHEST
298,Recon 2: ACRIN LARGE,,CHEST
...,...,...,...
929,,,CHEST
944,Recon 2: ACRIN LARGE,,CHEST
945,Recon 2: ACRIN LARGE,,CHEST
946,,,CHEST


In [83]:
generic_chest_screening_mask = (
    generic_chest_mask & screening_mask
)

generic_chest_screening_mask.sum()

np.int64(0)

In [84]:
low_dose_mask = (
    text_data["combined_text"].str.contains("LOW DOSE THROUGH LUNGS")
)

low_dose_mask.sum()

np.int64(8)

In [85]:
text_data[low_dose_mask][
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
590,Recon 2: LOW DOSE THROUGH LUNGS,,CHEST
599,Recon 2: LOW DOSE THROUGH LUNGS,,CHEST
604,Recon 2: LOW DOSE THROUGH LUNGS,,CT THORAX WO IV CONTRA
617,Recon 2: LOW DOSE THROUGH LUNGS,,CT THORAX WO IV CONTRA
628,Recon 2: LOW DOSE THROUGH LUNGS,,CHEST
646,Recon 2: LOW DOSE THROUGH LUNGS,,CHEST
662,Recon 2: LOW DOSE THROUGH LUNGS,,CT THORAX WO IV CONTRA
706,LOW DOSE THROUGH LUNGS,,CHEST


In [86]:
specific_category_mask = (
    screening_mask |
    noncontrast_mask |
    w_contrast_mask |
    cte_pe_mask |
    biopsy_mask |
    cap_expanded_mask
)

specific_category_mask.sum()

np.int64(338)

In [87]:
chest_mask = (
    specific_category_mask &
    generic_chest_mask
)

chest_mask.sum()

np.int64(0)

## Candidate Taxonomy Development

Exploratory review of the 573 usable CT series identified several recurring
protocol concepts and substantial variation in how those concepts were represented
across SeriesDescription, ProtocolName, and StudyDesc. Candidate groups included
lung screening, routine noncontrast chest CT, routine contrast-enhanced chest CT,
CT angiography/pulmonary embolism studies, CT-guided biopsy procedures, and
multiregion chest-abdomen-pelvis studies.

Boolean keyword masks were used for terminology exploration rather than as
reference labels. Overlapping terminology and generic descriptions demonstrated
that keyword matching alone may not provide sufficient evidence for reliable
classification. An UNKNOWN / INSUFFICIENT INFORMATION category will therefore
be retained for series that cannot be assigned confidently from the available
metadata.

## Reference Labeling Guidelines

Reference labels will be assigned through manual review of `SeriesDescription`, 
`ProtocolName`, and `StudyDesc`. The exploratory keyword masks developed above 
will be used to identify terminology patterns and organize records for review, 
but they will not automatically determine the reference label.

A series will be assigned to a protocol category only when the available metadata 
provides sufficient evidence to support that classification. When metadata are 
generic, incomplete, conflicting, or otherwise insufficient to distinguish among 
protocol categories, the series will be labeled `UNKNOWN / INSUFFICIENT INFORMATION`.

The following candidate categories and decision rules will be used during manual 
review.

### CT Angiography / Pulmonary Embolism

Assign `CTA_PE` when the available metadata explicitly identifies the series as 
chest CT angiography or a pulmonary embolism protocol. Supporting terminology 
may include `CT ANGIO CHEST`, `CHEST PE`, `CT CHEST PE`, or explicit `PE PROTOCOL` 
language. Generic contrast-enhanced chest terminology alone is not sufficient 
for this category.

### CT-Guided Biopsy / Procedure

Assign `CT_GUIDED_BIOPSY` when the metadata explicitly identifies a CT-guided 
lung biopsy or biopsy procedure. Supporting terminology may include `BIOPSY`, 
`LUNG BX`, or equivalent explicit biopsy terminology. Positioning terms such as 
`PRONE` or `SUPINE` are supportive only when accompanied by biopsy-related 
terminology and are not sufficient by themselves. Abbreviated or truncated descriptions such as `CT GUIDED LUNG B` are not
sufficient by themselves to establish a biopsy label unless another metadata
field provides explicit biopsy evidence.

### Routine Chest CT — Noncontrast

Assign `CHEST_NONCONTRAST` when the available metadata explicitly identifies a 
routine chest CT performed without intravenous contrast. Supporting terminology 
may include `WO CONTRAST`, `W/O CONTRAST`, `WO IV CONTRA`, or equivalent explicit 
noncontrast terminology. Generic descriptions such as `CHEST` or `CT CHEST` are 
not sufficient to establish contrast status.

### Routine Chest CT — Contrast

Assign `CHEST_CONTRAST` when the available metadata explicitly identifies a 
routine chest CT performed with intravenous contrast and does not indicate a more 
specific protocol category such as CT angiography or pulmonary embolism imaging. 
Supporting terminology may include `W/CONTRAST`, `W/ CONT`, `WITH IV CONTRAST`, 
or equivalent explicit contrast terminology. Generic chest terminology alone is 
not sufficient to establish contrast status.

### Lung Screening / Low-Dose Screening

Assign `LUNG_SCREENING` when the available metadata explicitly identifies the
series as a lung screening examination. Supporting terminology may include
`LUNG SCREEN`, `LUNG SCREENING`, `SCREENING CHEST`, `NLST`, or equivalent
explicit screening terminology.

Low-dose terminology alone is not sufficient to establish screening intent.
Descriptions such as `LOW DOSE THROUGH LUNGS` should therefore not automatically
receive the `LUNG_SCREENING` label unless another available metadata field
provides explicit evidence of screening. If the available metadata does not
support a more specific protocol category, the series should be assigned
`UNKNOWN / INSUFFICIENT INFORMATION`.

### Multiregion Chest-Abdomen-Pelvis

Assign `CHEST_ABDOMEN_PELVIS` when the available metadata explicitly identifies
a multiregion examination involving the chest, abdomen, and pelvis. Supporting
terminology may include `CAP`, `C-A-P`, `C/A/P`, `CHEST/ABD/PELVIS`,
`CH/ABD/PELVIS`, or equivalent terminology that clearly identifies all three
anatomic regions.

When fields provide conflicting information, such as multiregion terminology
in one field and a chest-only examination description in another, the record
should be manually reviewed rather than classified from a single keyword.
If the available metadata cannot resolve the conflict, assign
`UNKNOWN / INSUFFICIENT INFORMATION`.

### Unknown / Insufficient Information

Assign `UNKNOWN` when the available metadata does not provide sufficient evidence
to support one of the defined protocol categories. This includes records with
generic descriptions such as `CHEST`, missing descriptive fields, ambiguous
terminology, or conflicting information that cannot be resolved using the
available metadata.

The `UNKNOWN` category represents an intentional abstention rather than an
attempt to infer a protocol from unsupported information. Domain knowledge
should not be used to supply information that is absent from the metadata.

### Label Precedence and Conflict Resolution

When multiple metadata fields contain potentially relevant terminology, the
most specific explicitly supported protocol intent should take precedence over
a more general description. For example, explicit pulmonary embolism terminology
takes precedence over generic contrast-enhanced chest terminology, and explicit
lung screening terminology takes precedence over generic noncontrast chest
terminology.

Labels should be based on the combined evidence available in
`SeriesDescription`, `ProtocolName`, and `StudyDesc` rather than on a single
field whenever multiple fields contain useful information. Conflicting fields
should be manually reviewed. If the conflict cannot be resolved from the
available metadata, the series should be labeled `UNKNOWN`.

In [88]:
text_data["reference_label"] = pd.NA

text_data["reference_label"].isna().sum()

np.int64(573)

In [89]:
text_data[cte_pe_mask][["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
93,BOTTOM TO TOP,,CT ANGIO CHEST W&W/O C,<NA>
112,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
113,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST W&W/O C,<NA>
114,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
118,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
125,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
129,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
130,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
134,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>
136,BOTTOM TO TOP,5.7 @ CHEST PE BOTTOM TO TOP @,CT ANGIO CHEST (NON-CO,<NA>


In [90]:
text_data.loc[cte_pe_mask, "reference_label"] = "CTA_PE"

(text_data["reference_label"] == "CTA_PE").sum()

np.int64(25)

In [91]:
text_data.loc[
    biopsy_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
147,,8.2 LUNG BX PRONE,CT GUIDED LUNG B,<NA>
587,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,<NA>
591,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,<NA>
594,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,<NA>
597,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,<NA>
605,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,<NA>
609,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,<NA>
611,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG BIOPSY,<NA>
1000,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,<NA>
1001,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,<NA>


In [92]:
text_data.loc[biopsy_mask, "reference_label"] = "BIOPSY"

(text_data["reference_label"] == "BIOPSY").sum()

np.int64(11)

In [93]:
text_data.loc[
    text_data.index.isin([1000, 1001, 1008]),
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
1000,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,BIOPSY
1001,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,BIOPSY
1008,,5.13 LUNG SCREEN,CT GUIDED LUNG BIOPSY,BIOPSY


In [94]:
# Assignment — changes data
text_data.loc[[1000, 1001, 1008], "reference_label"] = "UNKNOWN"

# Retrieval — displays data
text_data.loc[[1000, 1001, 1008], ["SeriesDescription", "ProtocolName",
                     "StudyDesc", "reference_label"]]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
1000,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,UNKNOWN
1001,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY,UNKNOWN
1008,,5.13 LUNG SCREEN,CT GUIDED LUNG BIOPSY,UNKNOWN


In [95]:
text_data.loc[
    text_data["reference_label"] == "BIOPSY",
    "reference_label"
] = "CT_GUIDED_BIOPSY"

(text_data["reference_label"] == "CT_GUIDED_BIOPSY").sum()

np.int64(8)

In [96]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

In [97]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [98]:
Path("data/processed/dicom_reference_labels.csv").exists()

True

In [99]:
saved_labels = pd.read_csv(
    "data/processed/dicom_reference_labels.csv",
    index_col=0
)

saved_labels.shape

(573, 5)

In [100]:
saved_labels["reference_label"].value_counts(dropna=False)

reference_label
NaN                 537
CTA_PE               25
CT_GUIDED_BIOPSY      8
UNKNOWN               3
Name: count, dtype: int64

In [101]:
# Display the metatdata
text_data.loc[
    screening_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
146,,,CT LUNG SCREEN,<NA>
148,,,CT LUNG SCREEN,<NA>
149,,,CT LUNG SCREEN,<NA>
150,,,CT LUNG SCREEN,<NA>
151,,,CT LUNG SCREEN,<NA>
...,...,...,...,...
1011,,5.11 SCREENING CHEST,CT CHEST WO CONTRAST,<NA>
1014,,5.13 LUNG SCREEN,CT LUNG SCREEN,<NA>
1015,,5.13 LUNG SCREEN,CT LUNG SCREEN,<NA>
1016,,5.3 LUNG SCREENING,CT CHEST WO CONTRAST,<NA>


In [102]:
text_data.loc[screening_mask, "reference_label"].value_counts(dropna=False)

reference_label
<NA>       185
UNKNOWN      1
Name: count, dtype: int64

In [103]:
(text_data["reference_label"] == "SCREEN").sum()

np.int64(0)

In [104]:
screening_review = (
    text_data.loc[
        screening_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

screening_review

,SeriesDescription,ProtocolName,StudyDesc
146,,,CT LUNG SCREEN
175,,,e+1 CT LUNG SCREEN
452,NLST TLC VOL B30F,,
777,NLST_0.6_ B30f,,
955,,5.13 LUNG SCREEN,CT GUIDED LUNG B
957,,5.13 LUNG SCREEN,CT CHEST O CONTR
959,,5.13 LUNG SCREEN,CT LUNG SCREEN
961,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST
984,,5.11 SCREENING CHEST,CT CHEST WO CONTRAST
985,,5.11 SCREENING CHEST,CT LUNG SCREEN


In [105]:
biopsy_review = (
    text_data.loc[
        biopsy_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

biopsy_review

,SeriesDescription,ProtocolName,StudyDesc
147,,8.2 LUNG BX PRONE,CT GUIDED LUNG B
587,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B
594,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B
611,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG BIOPSY
1000,,5.11 FULL CHECT F/U,CT GUIDED LUNG BIOPSY
1008,,5.13 LUNG SCREEN,CT GUIDED LUNG BIOPSY


In [106]:
biopsy_review.shape

(6, 3)

In [107]:
text_data[
    text_data["StudyDesc"].str.contains("CT GUIDED", na=False)
][["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
147,,8.2 LUNG BX PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
587,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
588,,,CT GUIDED LUNG B,<NA>
589,,,CT GUIDED LUNG B,<NA>
591,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
592,,,CT GUIDED LUNG B,<NA>
594,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
597,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
605,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
606,,,CT GUIDED LUNG B,<NA>


In [108]:
text_data.loc[
    text_data["StudyDesc"].str.contains("CT GUIDED", na=False),
    "StudyDesc"
].value_counts()

StudyDesc
CT GUIDED LUNG B         22
CT GUIDED LUNG BIOPSY     4
Name: count, dtype: int64

In [109]:
text_data.loc[
    text_data["StudyDesc"] == "CT GUIDED LUNG B",
    ["ProtocolName", "StudyDesc"]
].value_counts(dropna=False)

ProtocolName          StudyDesc       
                      CT GUIDED LUNG B    8
5.20 LUNG BX. PRONE   CT GUIDED LUNG B    4
5.11 FULL CHECT F/U   CT GUIDED LUNG B    4
5.13 LUNG SCREEN      CT GUIDED LUNG B    3
5.19 LUNG BX. SUPINE  CT GUIDED LUNG B    2
8.2 LUNG BX PRONE     CT GUIDED LUNG B    1
Name: count, dtype: int64

### Biopsy Label Review

Initial review identified abbreviated and conflicting biopsy-related metadata.
Records containing biopsy terminology were therefore reviewed across
`SeriesDescription`, `ProtocolName`, and `StudyDesc` before assigning final
reference labels.

In [110]:
confirmed_biopsy_mask = (
    text_data["StudyDesc"].str.contains("CT GUIDED", na=False) &
    text_data["ProtocolName"].str.contains("BX", na=False)
)

confirmed_biopsy_mask.sum()

np.int64(8)

In [111]:
text_data.loc[
    confirmed_biopsy_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
147,,8.2 LUNG BX PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
587,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
591,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
594,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
597,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
605,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
609,,5.20 LUNG BX. PRONE,CT GUIDED LUNG B,CT_GUIDED_BIOPSY
611,,5.19 LUNG BX. SUPINE,CT GUIDED LUNG BIOPSY,CT_GUIDED_BIOPSY


In [112]:
(screening_mask & confirmed_biopsy_mask).sum()

np.int64(0)

In [113]:
screening_review = (
    text_data.loc[
        screening_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
    ]
    .drop_duplicates()
)

screening_review

,SeriesDescription,ProtocolName,StudyDesc,reference_label
146,,,CT LUNG SCREEN,<NA>
175,,,e+1 CT LUNG SCREEN,<NA>
452,NLST TLC VOL B30F,,,<NA>
777,NLST_0.6_ B30f,,,<NA>
955,,5.13 LUNG SCREEN,CT GUIDED LUNG B,<NA>
957,,5.13 LUNG SCREEN,CT CHEST O CONTR,<NA>
959,,5.13 LUNG SCREEN,CT LUNG SCREEN,<NA>
961,,5.13 LUNG SCREEN,CT CHEST WO CONTRAST,<NA>
984,,5.11 SCREENING CHEST,CT CHEST WO CONTRAST,<NA>
985,,5.11 SCREENING CHEST,CT LUNG SCREEN,<NA>


In [114]:
confirmed_screening_mask = (
    screening_mask &
    ~text_data["StudyDesc"].str.contains("CT GUIDED", na=False)
)

confirmed_screening_mask.sum()

np.int64(182)

In [115]:
screening_conflict_mask = (
    screening_mask & 
    text_data["StudyDesc"].str.contains("CT GUIDED", na=False)
)

screening_conflict_mask.sum()

np.int64(4)

In [116]:
text_data.loc[
    screening_conflict_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
955,,5.13 LUNG SCREEN,CT GUIDED LUNG B,<NA>
1002,,5.13 LUNG SCREEN,CT GUIDED LUNG B,<NA>
1005,,5.13 LUNG SCREEN,CT GUIDED LUNG B,<NA>
1008,,5.13 LUNG SCREEN,CT GUIDED LUNG BIOPSY,UNKNOWN


In [117]:
text_data.loc[
    screening_conflict_mask,
    "reference_label",
] = "UNKNOWN"

In [118]:
text_data.loc[
    screening_conflict_mask,
    "reference_label",
].value_counts()

reference_label
UNKNOWN    4
Name: count, dtype: int64

In [119]:
text_data.loc[
    confirmed_screening_mask,
    "reference_label",
] = "LUNG SCREENING"

In [120]:
text_data.loc[
    confirmed_screening_mask,
    "reference_label",
].value_counts()

reference_label
LUNG SCREENING    182
Name: count, dtype: int64

In [121]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [122]:
text_data["reference_label"].value_counts(dropna=False)

reference_label
<NA>                352
LUNG SCREENING      182
CTA_PE               25
CT_GUIDED_BIOPSY      8
UNKNOWN               6
Name: count, dtype: int64

In [123]:
noncontrast_mask.sum()

np.int64(19)

In [124]:
(noncontrast_mask & screening_mask).sum()

np.int64(9)

In [125]:
text_data.loc[
    noncontrast_mask,
    "reference_label"
].value_counts(dropna=False)

reference_label
<NA>              10
LUNG SCREENING     9
Name: count, dtype: int64

In [126]:
noncontrast_unlabeled_mask = (
    noncontrast_mask &
    text_data["reference_label"].isna()
)

noncontrast_unlabeled_mask.sum()

np.int64(10)

In [127]:
text_data.loc[
    noncontrast_unlabeled_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
958,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
960,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
962,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
977,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
978,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
980,,5.11 FULL CHECT F/U,CT CHEST WO CONTRAST,<NA>
983,,5.10 Full Chest F/U,CT CHEST WO CONTRAST,<NA>
986,,5.10 Full Chest F/U,CT CHEST WO CONTRAST,<NA>
987,,5.10 Full Chest F/U,CT CHEST WO CONTRAST,<NA>
1012,,5.10 Full Chest F/U,CT CHEST WO CONTRAST,<NA>


In [128]:
text_data.loc[
    noncontrast_unlabeled_mask,
    "reference_label",
] = "WO CONTRAST"

In [129]:
text_data.loc[
    noncontrast_unlabeled_mask,
    "reference_label",
].value_counts()

reference_label
WO CONTRAST    10
Name: count, dtype: int64

In [130]:
text_data.loc[
    noncontrast_unlabeled_mask,
    "reference_label"
] = "CHEST_NONCONTRAST"

In [131]:
(text_data["reference_label"] == "CHEST_NONCONTRAST").sum()

np.int64(10)

In [132]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [133]:
w_contrast_mask.sum()

np.int64(72)

In [134]:
text_data.loc[
    w_contrast_mask,
    "reference_label",
].value_counts(dropna=False)

reference_label
<NA>    72
Name: count, dtype: int64

In [135]:
w_contrast_unlabeled_mask = (
    w_contrast_mask &
    text_data["reference_label"].isna()
)

w_contrast_unlabeled_mask.sum()

np.int64(72)

In [136]:
text_data.loc[
    w_contrast_unlabeled_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
]

,SeriesDescription,ProtocolName,StudyDesc,reference_label
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
28,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
29,Recon 2: CHEST,,CT THORAX W/CONTRAST,<NA>
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST,<NA>
35,,,CT THORAX W/CONTRAST,<NA>
...,...,...,...,...
409,Recon 3: C-A-P,5.3 @ C-A-P @,CT THORAX W/CONTRAST,<NA>
413,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
419,CHEST,5.7 @ CHEST HEAD FIRST @,CT THORAX W/CONTRAST,<NA>
421,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>


In [137]:
w_contrast_review = (
    text_data.loc[
        w_contrast_unlabeled_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc", "reference_label"]
    ].drop_duplicates()
)

w_contrast_review

,SeriesDescription,ProtocolName,StudyDesc,reference_label
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
29,Recon 2: CHEST,,CT THORAX W/CONTRAST,<NA>
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST,<NA>
35,,,CT THORAX W/CONTRAST,<NA>
41,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT THORAX W/CONTRAST,<NA>
43,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
48,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,CT THORAX W/CONTRAST,<NA>
54,,5.1 @ CHEST @,CT THORAX W/CONTRAST,<NA>
55,,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST,<NA>
104,Recon 2: C-A-P,,CT THORAX W/CONTRAST,<NA>


In [138]:
w_contrast_review.shape

(22, 4)

In [139]:
(w_contrast_unlabeled_mask & cap_expanded_mask).sum()

np.int64(8)

In [140]:
(w_contrast_unlabeled_mask & ~cap_expanded_mask).sum()

np.int64(64)

In [141]:
contrast_noncap_mask = (
    w_contrast_unlabeled_mask &
    ~cap_expanded_mask
)

contrast_noncap_mask.sum()

np.int64(64)

In [142]:
contrast_noncap_review = (
    text_data.loc[
        contrast_noncap_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

contrast_noncap_review

,SeriesDescription,ProtocolName,StudyDesc
27,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
29,Recon 2: CHEST,,CT THORAX W/CONTRAST
30,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST
35,,,CT THORAX W/CONTRAST
43,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
54,,5.1 @ CHEST @,CT THORAX W/CONTRAST
55,,5.1 @ CHEST 5mm @,CT THORAX W/CONTRAST
109,CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
110,Recon 2: CHEST,5.1 @ CHEST @,CT THORAX W/CONTRAST
115,CHEST,,CT THORAX W/CONTRAST


In [143]:
contrast_noncap_review.shape

(15, 3)

In [144]:
text_data.loc[
    contrast_noncap_mask,
    "reference_label"
] = "CHEST_CONTRAST"

In [145]:
(text_data["reference_label"] == "CHEST_CONTRAST").sum()

np.int64(64)

In [146]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [147]:
cap_expanded_mask.sum()

np.int64(43)

In [148]:
text_data.loc[
    cap_expanded_mask,
    "reference_label",
].value_counts(dropna=False)

reference_label
<NA>    43
Name: count, dtype: int64

In [149]:
cap_unlabeled_mask = (
    cap_expanded_mask &
    text_data["reference_label"].isna()
)

cap_unlabeled_mask.sum()

np.int64(43)

In [150]:
text_data.loc[
    cap_unlabeled_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]    

,SeriesDescription,ProtocolName,StudyDesc
26,Recon 2: C-A-P,,CT/C-A-P
34,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT CH/AB/PELVIS
36,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CH/ABD/PELVIS
37,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,T/A/P W/C
38,,,CT CH/ABD/PELVIS
41,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT THORAX W/CONTRAST
46,Recon 2: C-A-P,5.3 @ C-A-P 5mm @,CT-CH/A/P
48,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,CT THORAX W/CONTRAST
50,Recon 2: C-A-P,5.4 @ C-A-P LYM. HEAD FIRST @,C/A/P POST
52,,5.3 @ C-A-P 5mm @,CHEST/ABD/PELVIS


In [151]:
cap_specific_overlap = (
    cap_unlabeled_mask &
    (
        screening_mask |
        cte_pe_mask |
        confirmed_biopsy_mask
    )
)

cap_specific_overlap.sum()

np.int64(0)

In [152]:
text_data.loc[
    cap_unlabeled_mask,
    "reference_label"
] = "CAP"

In [153]:
(text_data["reference_label"] == "CAP").sum()

np.int64(43)

In [154]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [155]:
unlabeled_mask = text_data["reference_label"].isna()

unlabeled_mask.sum()

np.int64(235)

In [156]:
unlabeled_review = (
    text_data.loc[
        unlabeled_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

unlabeled_review.shape

(62, 3)

In [157]:
unlabeled_pattern_counts = (
    text_data.loc[
        unlabeled_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .value_counts(dropna=False)
)

unlabeled_pattern_counts.head(20)

SeriesDescription                ProtocolName         StudyDesc             
Recon 2: ACRIN LARGE                                  CHEST                     30
ThoraxRoutine  3.0  B31s                                                        24
                                                      CHEST                     21
                                 5.11 FULL CHECT F/U  CT CHEST O CONTR          16
                                                      CT CHEST O CONTR          13
Recon 2:                                              CHEST                     13
Recon 2: ACRIN LARGE                                  CT THORAX WO IV CONTRA    13
ChestRoutine  3.0  B31f                                                         12
ThorRoutine  2.0  B70f                                                           8
                                                      CT GUIDED LUNG B           8
ThorRoutine  3.0  B31f                                                           6
          

## Review of Unlabeled Records and Keyword Rule Refinement

After initial reference labeling, 235 of the 573 CT series remained unlabeled. 
Review of the remaining records showed that some series represented protocol 
categories already included in the candidate taxonomy but used terminology that 
was not captured by the initial Boolean keyword masks.

For example, additional noncontrast chest terminology included variants such as 
"CT THORAX WO IV CONTRA," "CT THORAX W/O CONTRAST," and "CT CHEST O CONTR." 
These variations demonstrate a limitation of exact keyword matching and support 
the need to iteratively review and refine terminology rules.

The remaining unlabeled records were therefore examined for recurring metadata 
patterns. Existing masks were revisited when additional terminology variants 
were identified. Reference labels were assigned only after reviewing the 
available SeriesDescription, ProtocolName, and StudyDesc fields. Records with 
insufficient or conflicting evidence were retained for further review or 
assigned UNKNOWN when a reliable classification could not be made.

In [158]:
unlabeled_contrast_term_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains("CONTR", na=False)
)

unlabeled_contrast_term_mask.sum()    

np.int64(65)

In [159]:
unlabeled_contrast_review = (
    text_data.loc[
        unlabeled_contrast_term_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

unlabeled_contrast_review

,SeriesDescription,ProtocolName,StudyDesc
32,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/O CONTRAST
33,,,CT THORAX W/O CONTRAST
98,Recon 3: CHEST,5.1 ROUTINE CHEST WITH IV CONTRAST 100,CT CHEST W/ CONT
116,Recon 2: CHEST,,CT THORAX W/O CONTRAST
128,,5.1 ROUTINE CHEST WITH IV CONTRAST 100,CT CHEST W/ CONT
246,Recon 3: CT CHEST,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/ CONT
264,,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/O CON
267,,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/O CON
273,1.25 THROUGH LUNGS,,CT THORAX WO IV CONTRA
307,135cc 4cc/sec,,CT THORAX W IV CONTRAS


In [160]:
unlabeled_contrast_review.shape

(20, 3)

### Handling Differences Across Metadata Fields

Review of the remaining unlabeled records identified cases in which DICOM metadata fields differed in specificity. For example, some ProtocolName values described a routine chest protocol as "WITH OR WITHOUT CONTRAST," while the corresponding StudyDesc identified the individual examination as "W/O CON." In these cases, the more specific description of the individual study was used when assigning the reference label rather than inferring contrast status from the broader protocol name.

Contrast terminology was also represented using several abbreviations, including "W/CONTRAST," "W IV CONTRAS," "W/O CONTRAST," "WO IV CONTRA," and "O CONTR." These variations were reviewed before refining the Boolean masks so that additional terminology could be incorporated without automatically treating every occurrence of "CONTR" as evidence of contrast administration.

In [161]:
with_contrast_terms = (
    text_data["combined_text"].str.contains(
        r"W/ ?CONT|WITH IV CONTRAST|W IV CONTRAS",
        na=False
    )
)

In [162]:
residual_with_contrast_mask = (
    unlabeled_mask &
    with_contrast_terms
)

residual_with_contrast_mask.sum()

np.int64(7)

In [163]:
without_contrast_terms = (
    text_data["combined_text"].str.contains(
        r"W/O ?CONTRAST|WO IV CONTRA|W/O ?CON|O CONTR",
        na=False
    )
)

In [164]:
residual_without_contrast_mask = (
    unlabeled_mask &
    without_contrast_terms
)

residual_without_contrast_mask.sum()

np.int64(63)

In [165]:
(
    residual_with_contrast_mask &
    residual_without_contrast_mask
).sum()

np.int64(0)

In [166]:
residual_with_contrast_mask.sum()

residual_without_contrast_mask.sum()

(
    residual_with_contrast_mask &
    residual_without_contrast_mask
).sum()

np.int64(0)

In [167]:
text_data.loc[
    residual_with_contrast_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
88,Recon 3:,5.1 ROUTINE CHEST WITH,CT CHEST W/ CONT
98,Recon 3: CHEST,5.1 ROUTINE CHEST WITH IV CONTRAST 100,CT CHEST W/ CONT
128,,5.1 ROUTINE CHEST WITH IV CONTRAST 100,CT CHEST W/ CONT
246,Recon 3: CT CHEST,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/ CONT
253,CT CHEST 5 X 5,5.2 CT #101 CHEST LUNG CA. STAGING*,CT CHEST W/ CONT
307,135cc 4cc/sec,,CT THORAX W IV CONTRAS
644,,,CT THORAX W IV CONTRAS


In [168]:
text_data.loc[
    residual_with_contrast_mask,
    "reference_label"
] = "CHEST_CONTRAST"

In [169]:
(text_data["reference_label"] == "CHEST_CONTRAST").sum()

np.int64(71)

In [178]:
unlabeled_mask = text_data["reference_label"].isna()

In [179]:
without_contrast_terms = (
    text_data["combined_text"].str.contains(
        r"W/O ?CONTRAST|WO IV CONTRA|W/O ?CON|O CONTR",
        na=False
    )
)

In [181]:
residual_without_contrast_mask = (
    unlabeled_mask &
    without_contrast_terms
)

residual_without_contrast_mask.sum()

np.int64(63)

In [182]:
residual_without_contrast_review = (
    text_data.loc[
        residual_without_contrast_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

residual_without_contrast_review.shape

(18, 3)

In [183]:
residual_without_contrast_review

,SeriesDescription,ProtocolName,StudyDesc
32,Recon 2: CHEST,5.1 @ CHEST 5mm @,CT THORAX W/O CONTRAST
33,,,CT THORAX W/O CONTRAST
87,1.25 STANDARD,5.2 (100)CT CHEST WITHOUT,CT CHEST W/O CON
95,Recon 3: CHEST 5X5,5.2 CT CHEST WITHOUT,CT CHEST W/O CON
116,Recon 2: CHEST,,CT THORAX W/O CONTRAST
126,,5.2 ROUTINE CHEST WITHOUT 100,CT CHEST W/O CON
264,,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/O CON
267,,5.1 CT #100 ROUTINE CHEST WITH OR WITHOUT CONT...,CT CHEST W/O CON
273,1.25 THROUGH LUNGS,,CT THORAX WO IV CONTRA
369,CHEST,5.1 @ CHEST @,CT THORAX W/O CONTRAST


In [185]:
text_data.loc[
    residual_without_contrast_mask,
    "reference_label"
] = "CHEST_NONCONTRAST"

In [186]:
(text_data["reference_label"] == "CHEST_NONCONTRAST").sum()

np.int64(73)

In [187]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [188]:
unlabeled_mask = text_data["reference_label"].isna()

unlabeled_mask.sum()

np.int64(165)

In [189]:
remaining_pattern_counts = (
    text_data.loc[
        unlabeled_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .value_counts(dropna=False)
)

remaining_pattern_counts.head(20)

SeriesDescription                ProtocolName           StudyDesc            
Recon 2: ACRIN LARGE                                    CHEST                    30
ThoraxRoutine  3.0  B31s                                                         24
                                                        CHEST                    21
Recon 2:                                                CHEST                    13
ChestRoutine  3.0  B31f                                                          12
ThorRoutine  2.0  B70f                                                            8
                                                        CT GUIDED LUNG B          8
ThorRoutine  3.0  B31f                                                            6
Recon 2: LOW DOSE THROUGH LUNGS                         CHEST                     4
ACRIN LARGE                                             CHEST                     4
                                 5.11 FULL CHECT F/U    CT GUIDED LUNG B          

In [190]:
remaining_ct_guided_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains(
        "CT GUIDED",
        na=False
    )
)

remaining_ct_guided_mask.sum()

np.int64(12)

In [191]:
text_data.loc[
    remaining_ct_guided_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
].drop_duplicates()

,SeriesDescription,ProtocolName,StudyDesc
588,,,CT GUIDED LUNG B
956,,5.11 FULL CHECT F/U,CT GUIDED LUNG B


In [192]:
text_data.loc[
    remaining_ct_guided_mask,
    "reference_label"
] = "UNKNOWN"

In [193]:
(text_data["reference_label"] == "UNKNOWN").sum()

np.int64(18)

In [194]:
unlabeled_mask = text_data["reference_label"].isna()

unlabeled_mask.sum()

np.int64(153)

### Ambiguous CT-Guided Procedure Metadata

Twelve remaining records contained the truncated StudyDesc "CT GUIDED LUNG B" but lacked additional terminology sufficient to confirm a biopsy procedure. Earlier records were assigned CT_GUIDED_BIOPSY when complementary metadata such as "BX," "BX PRONE," or "BX SUPINE" supported that interpretation. The truncated description alone was not treated as sufficient evidence because doing so would require inferring information not explicitly represented in the available metadata. These records were therefore assigned UNKNOWN.

In [195]:
text_data.to_csv(
    "data/processed/dicom_reference_labels.csv",
    index=True
)

In [196]:
generic_chest_candidate_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains(
        r"CHEST|THORAX|THOR",
        na=False
    )
)

generic_chest_candidate_mask.sum()

np.int64(87)

In [197]:
generic_chest_review = (
    text_data.loc[
        generic_chest_candidate_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

generic_chest_review

,SeriesDescription,ProtocolName,StudyDesc
31,Recon 2: CHEST,,"CT, LUNG, MED, PLEURA"
40,Recon 2: CHEST-ABD,5.2 @ CHEST-ABD 5mm @,CT C/A
121,Recon 2: CHEST & BLADDER,,CT-CH/A/P LIVER
216,,,CHEST
270,,5.1 @ CHEST @,POST CHEST
289,,,CT CHEST NO IV
295,Recon 2: ACRIN LARGE,,CHEST
300,Recon 2: ACRIN SMALL,,CT CHEST
308,Recon 2: ACRIN LARGE,,CHEST172
361,Recon 3: CHEST-ABD,5.5 @ CHEST-ABD @,CT C/A


In [198]:
generic_chest_review.shape

(20, 3)

In [199]:
(
    generic_chest_candidate_mask &
    cap_expanded_mask
).sum()

np.int64(0)

In [200]:
text_data.loc[
    generic_chest_candidate_mask & cap_expanded_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
].drop_duplicates()

,SeriesDescription,ProtocolName,StudyDesc


In [201]:
residual_multiregion_terms = (
    text_data["combined_text"].str.contains(
        r"CHEST-ABD|CHEST/ABD|CH-AB|CT C/A|CH/A/P",
        na=False
    )
)

In [202]:
residual_multiregion_mask = (
    unlabeled_mask &
    generic_chest_candidate_mask &
    residual_multiregion_terms
)

residual_multiregion_mask.sum()

np.int64(5)

In [203]:
text_data.loc[
    residual_multiregion_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
].drop_duplicates()

,SeriesDescription,ProtocolName,StudyDesc
40,Recon 2: CHEST-ABD,5.2 @ CHEST-ABD 5mm @,CT C/A
121,Recon 2: CHEST & BLADDER,,CT-CH/A/P LIVER
361,Recon 3: CHEST-ABD,5.5 @ CHEST-ABD @,CT C/A
389,CHEST-ABD,5.5 @ CHEST-ABD @,CHEST/ABD W/C
399,Recon 3: CHEST-ABD,5.5 @ CHEST-ABD @,CH-AB W/WO


In [204]:
text_data.loc[
    residual_multiregion_mask,
    "reference_label"
] = "UNKNOWN"

In [205]:
(text_data["reference_label"] == "UNKNOWN").sum()

np.int64(23)

### Out-of-Taxonomy Multiregion Studies

Residual review identified five multiregion studies containing combinations such as
"CHEST-ABD," "CT C/A," "CHEST/ABD W/C," and "CT-CH/A/P LIVER." Although the
available metadata provided information about the anatomical regions examined,
these studies did not fit consistently within the predefined reference taxonomy.
Additional categories were not created for these small residual groups because
doing so would increase taxonomy complexity while producing classes with very
few observations. These records were therefore assigned UNKNOWN, which includes
records with insufficient, conflicting, or out-of-taxonomy metadata.

In [206]:
unlabeled_mask = text_data["reference_label"].isna()

unlabeled_mask.sum()

np.int64(148)

In [207]:
generic_chest_candidate_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains(
        r"CHEST|THORAX|THOR",
        na=False
    )
)

generic_chest_candidate_mask.sum()

np.int64(82)

In [208]:
generic_chest_review = (
    text_data.loc[
        generic_chest_candidate_mask,
        ["SeriesDescription", "ProtocolName", "StudyDesc"]
    ]
    .drop_duplicates()
)

generic_chest_review

,SeriesDescription,ProtocolName,StudyDesc
31,Recon 2: CHEST,,"CT, LUNG, MED, PLEURA"
216,,,CHEST
270,,5.1 @ CHEST @,POST CHEST
289,,,CT CHEST NO IV
295,Recon 2: ACRIN LARGE,,CHEST
300,Recon 2: ACRIN SMALL,,CT CHEST
308,Recon 2: ACRIN LARGE,,CHEST172
366,CHEST,5.1 @ CHEST @,THORAX WITH
414,CHEST,5.1 @ CHEST @,POST CHEST
415,CHEST,5.7 @ CHEST HEAD FIRST @,PRE CHEST


In [209]:
generic_chest_review.shape

(15, 3)

Chest CT is identifiable from the available metadata, but the metadata does not provide sufficient evidence to assign a more specific chest protocol category

In [210]:
no_iv_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains(
        r"NO IV",
        na=False
    )
)

no_iv_mask.sum()

np.int64(1)

In [211]:
text_data.loc[
    no_iv_mask,
    ["SeriesDescription", "ProtocolName", "StudyDesc"]
]

,SeriesDescription,ProtocolName,StudyDesc
289,,,CT CHEST NO IV


In [213]:
text_data.loc[
    no_iv_mask,
    "reference_label"
] = "CHEST_NONCONTRAST"

In [214]:
(text_data["reference_label"] == "CHEST_NONCONTRAST").sum()

np.int64(74)

In [215]:
unlabeled_mask = text_data["reference_label"].isna()

unlabeled_mask.sum()

np.int64(147)

In [216]:
generic_chest_candidate_mask = (
    unlabeled_mask &
    text_data["combined_text"].str.contains(
        r"CHEST|THORAX|THOR",
        na=False
    )
)

generic_chest_candidate_mask.sum()

np.int64(81)

In [217]:
specific_overlap_mask = (
    generic_chest_candidate_mask &
    (
        screening_mask |
        cte_pe_mask |
        confirmed_biopsy_mask |
        cap_expanded_mask
    )
)

specific_overlap_mask.sum()

np.int64(0)

### Generic Chest CT Classification

Residual review identified CT series in which the available metadata clearly
supported the chest or thorax as the anatomical region but did not provide
sufficient information to determine contrast status or another specific protocol
class. Examples included generic descriptions such as "CHEST," "CT CHEST,"
"ThoraxRoutine," and reconstruction terminology associated with chest imaging.

These records were assigned CHEST_UNSPECIFIED rather than UNKNOWN because the
anatomical examination could be identified reliably even though a more specific
protocol classification was not supported by the available metadata. Contrast
status was not inferred from ambiguous terms such as "PRE," "POST," or incomplete
descriptions such as "THORAX WITH."

In [218]:
text_data.loc[
    generic_chest_candidate_mask,
    "reference_label"
] = "CHEST_UNSPECIFIED"

In [219]:
(text_data["reference_label"] == "CHEST_UNSPECIFIED").sum()

np.int64(81)

In [220]:
unlabeled_mask = text_data["reference_label"].isna()
unlabeled_mask.sum()

np.int64(66)